# Notebook 6 (Fase 1) - LSTM Regressor -> MongoDB
### Ernesto Investing AI . iDeSo . UNMSM -- Semana 13

**Objetivo:**

Entrenar una red neuronal recurrente **LSTM Regressor** para pronosticar el precio de cierre de los 5 tickers mineros del proyecto, contrastarla contra un modelo **ARIMA(1,1,1)** de control, y almacenar las predicciones (horizontes de 7, 14, 30 y 60 dias) junto con las metricas del modelo en MongoDB Atlas.

**Prerequisito:** Haber ejecutado el Notebook 1 (coleccion `precios_ohlcv` poblada). Si algun ticker no tiene suficientes datos, este notebook los descarga automaticamente con `yfinance`.

**Salida:** Colecciones `predicciones_lstm` y `metricas_lstm` pobladas para los 5 tickers, y archivo local `datos_lstm_reg.json` con el consolidado.

**Pipeline:** yfinance -> Ingesta (`precios_ohlcv`) -> LSTM Regressor & ARIMA -> MongoDB (`predicciones_lstm`, `metricas_lstm`) -> `datos_lstm_reg.json`

In [1]:
# Paso 1 -- Instalar librerias necesarias
!pip install "pymongo[srv]" yfinance scikit-learn tensorflow statsmodels pandas --quiet

print("Librerias instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.3 MB/s eta 0:00:00
Librerias instaladas correctamente


In [2]:
# Paso 2 -- Conectar a MongoDB Atlas
# La contrasena NUNCA se escribe en texto plano en el notebook: se pide de forma
# oculta con getpass y se inserta en la URI de conexion en tiempo de ejecucion.
# Alternativa con Secrets de Colab:
# from google.colab import userdata; MONGO_URI = userdata.get("MONGO_URI")
from getpass import getpass
from pymongo import MongoClient
from pymongo.server_api import ServerApi

DB_USER = "gdev03_db_user"
DB_PASSWORD = getpass("Password de MongoDB Atlas: ")

MONGO_URI = (
    f"mongodb+srv://{DB_USER}:{DB_PASSWORD}"
    "@cluster0.sxjck8h.mongodb.net/?appName=Cluster0"
)

client = MongoClient(MONGO_URI, server_api=ServerApi("1"))

# Verificar la conexion con un ping, manejando errores de forma explicita
try:
    client.admin.command("ping")
    print("Conectado a MongoDB Atlas correctamente")
except Exception as e:
    raise Exception("Ocurrio el siguiente error al conectar: ", e)

db = client["ernesto_investing_ai"]

Password de MongoDB Atlas: ··········
Conectado a MongoDB Atlas correctamente


In [3]:
# Paso 3 -- Definir los 5 tickers mineros del proyecto
TICKERS = {
    "FSM": "Fortuna Silver Mines Inc.",
    "VOLCABC1.LM": "Volcan Compania Minera S.A.A.",
    "ABX.TO": "Barrick Gold Corporation",
    "BVN": "Compania de Minas Buenaventura",
    "BHP": "BHP Billiton Limited"
}

print("Tickers del proyecto:")
for t, nombre in TICKERS.items():
    print(f"  {t}: {nombre}")

Tickers del proyecto:
  FSM: Fortuna Silver Mines Inc.
  VOLCABC1.LM: Volcan Compania Minera S.A.A.
  ABX.TO: Barrick Gold Corporation
  BVN: Compania de Minas Buenaventura
  BHP: BHP Billiton Limited


In [4]:
# Paso 4 -- Importaciones generales y funciones de indicadores tecnicos
# (mismas funciones del Notebook 1, replicadas aqui para que este notebook
# sea autocontenido y no dependa de reejecutar el Notebook 1).
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta

def calcular_sma(serie, periodo):
    """Media movil simple."""
    return serie.rolling(window=periodo).mean()

def calcular_ema(serie, periodo):
    """Media movil exponencial."""
    return serie.ewm(span=periodo, adjust=False).mean()

def calcular_rsi(serie, periodo=14):
    """Indice de fuerza relativa (RSI)."""
    delta = serie.diff()
    ganancia = (delta.where(delta > 0, 0)).rolling(window=periodo).mean()
    perdida = (-delta.where(delta < 0, 0)).rolling(window=periodo).mean()
    rs = ganancia / perdida
    return 100 - (100 / (1 + rs))

def limpiar_valor(v):
    """Reemplaza NaN/None por null valido para JSON/MongoDB, y castea a float."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return round(float(v), 4)

print("Funciones de indicadores tecnicos y utilidades definidas")

Funciones de indicadores tecnicos y utilidades definidas


In [5]:
# Paso 5 -- Garantizar datos suficientes por ticker (consulta MongoDB o descarga con yfinance)
MIN_REGISTROS = 150

def ingestar_ticker_yfinance(ticker, periodo="1y"):
    """Descarga OHLCV de yfinance, calcula los mismos indicadores tecnicos del
    Notebook 1 (SMA-20, SMA-50, EMA-12, EMA-26, RSI-14) y los guarda en
    MongoDB, garantizando consistencia entre colecciones.
    """
    df = yf.download(ticker, period=periodo, auto_adjust=True, progress=False)

    if df.empty:
        raise ValueError(f"yfinance no devolvio datos para {ticker}")

    # yfinance devuelve un MultiIndex en las columnas incluso para un solo ticker
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df["sma_20"] = calcular_sma(df["Close"], 20)
    df["sma_50"] = calcular_sma(df["Close"], 50)
    df["ema_12"] = calcular_ema(df["Close"], 12)
    df["ema_26"] = calcular_ema(df["Close"], 26)
    df["rsi_14"] = calcular_rsi(df["Close"], 14)

    registros = []
    for fecha, fila in df.iterrows():
        registros.append({
            "ticker": ticker,
            "fecha": fecha.strftime("%Y-%m-%d"),
            "open": limpiar_valor(fila["Open"]),
            "high": limpiar_valor(fila["High"]),
            "low": limpiar_valor(fila["Low"]),
            "close": limpiar_valor(fila["Close"]),
            "volume": int(fila["Volume"]) if not pd.isna(fila["Volume"]) else 0,
            "sma_20": limpiar_valor(fila["sma_20"]),
            "sma_50": limpiar_valor(fila["sma_50"]),
            "ema_12": limpiar_valor(fila["ema_12"]),
            "ema_26": limpiar_valor(fila["ema_26"]),
            "rsi_14": limpiar_valor(fila["rsi_14"]),
            "created_at": datetime.now()
        })

    # Se borran registros previos del ticker para evitar duplicados al re-ejecutar
    db["precios_ohlcv"].delete_many({"ticker": ticker})
    db["precios_ohlcv"].insert_many(registros)
    return len(registros)


def garantizar_datos_ticker(ticker):
    """Verifica que existan al menos MIN_REGISTROS documentos en `precios_ohlcv`
    para el ticker. Si no es asi, descarga 1 anio con yfinance y los guarda.
    Devuelve la cantidad final de registros disponibles.
    """
    n = db["precios_ohlcv"].count_documents({"ticker": ticker})
    if n >= MIN_REGISTROS:
        print(f"  [{ticker}] {n} registros ya disponibles en MongoDB")
        return n

    print(f"  [{ticker}] Solo {n} registros en MongoDB (< {MIN_REGISTROS}), descargando con yfinance...")
    n_nuevos = ingestar_ticker_yfinance(ticker, periodo="1y")
    print(f"  [{ticker}] {n_nuevos} registros descargados y guardados en MongoDB")
    return n_nuevos


def cargar_serie_close(ticker):
    """Lee de MongoDB la serie temporal ordenada de precios de cierre de un ticker."""
    docs = list(
        db["precios_ohlcv"].find({"ticker": ticker}, {"_id": 0, "fecha": 1, "close": 1})
        .sort("fecha", 1)
    )
    df = pd.DataFrame(docs).dropna(subset=["close"]).reset_index(drop=True)
    df["fecha"] = pd.to_datetime(df["fecha"])
    return df

print("Funciones de ingesta condicional y carga de series definidas")

Funciones de ingesta condicional y carga de series definidas


In [6]:
# Paso 6 -- Preprocesamiento: particion 80/20, normalizacion y ventanas deslizantes
from sklearn.preprocessing import MinMaxScaler

VENTANA = 60

def particionar_train_test(serie_close, prop_train=0.8):
    """Particiona la serie de forma TEMPORAL (no aleatoria): el 80% inicial
    de las fechas es train y el 20% final es test, respetando el orden
    cronologico para no filtrar informacion futura al entrenamiento.
    """
    n_train = int(len(serie_close) * prop_train)
    train = serie_close[:n_train]
    test = serie_close[n_train:]
    return train, test


def normalizar(train, test):
    """Ajusta el MinMaxScaler UNICAMENTE con el set de entrenamiento (para
    evitar data leakage) y lo aplica tanto a train como a test.
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train.reshape(-1, 1))
    test_norm = scaler.transform(test.reshape(-1, 1))
    return train_norm, test_norm, scaler


def generar_ventanas(serie_norm, ventana=VENTANA):
    """Genera ventanas deslizantes de tamanio `ventana` (X) para predecir el
    valor inmediatamente siguiente (y). X queda con forma (muestras, ventana, 1).
    """
    X, y = [], []
    for i in range(ventana, len(serie_norm)):
        X.append(serie_norm[i - ventana:i, 0])
        y.append(serie_norm[i, 0])
    X = np.array(X).reshape(-1, ventana, 1)
    y = np.array(y)
    return X, y

print(f"Funciones de preprocesamiento definidas (ventana = {VENTANA} dias)")

Funciones de preprocesamiento definidas (ventana = 60 dias)


In [7]:
# Paso 7 -- Arquitectura del modelo LSTM Regressor (Keras/TensorFlow)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

def construir_modelo_lstm(ventana=VENTANA):
    """Construye la arquitectura LSTM Regressor especificada:

    LSTM(64, return_sequences=True) -> Dropout(0.2) ->
    LSTM(32, return_sequences=False) -> Dropout(0.2) ->
    Dense(16, activation="relu") -> Dense(1, activation="linear")
    """
    modelo = Sequential([
        LSTM(64, return_sequences=True, input_shape=(ventana, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear")
    ])
    modelo.compile(optimizer=Adam(), loss="mean_squared_error")
    return modelo

print("Funcion construir_modelo_lstm definida")

Funcion construir_modelo_lstm definida


In [8]:
# Paso 8 -- Entrenamiento del modelo (particion temporal 80/20, 100 epocas, batch_size=32)
def entrenar_modelo_lstm(modelo, X_train, y_train, X_test, y_test, epocas=100, batch_size=32):
    """Compila y entrena el modelo LSTM usando el set de test (obtenido de la
    particion TEMPORAL 80/20) como validacion durante el entrenamiento.
    """
    historial = modelo.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=epocas,
        batch_size=batch_size,
        verbose=0
    )
    return historial

print("Funcion entrenar_modelo_lstm definida")

Funcion entrenar_modelo_lstm definida


In [9]:
# Paso 9 -- Pronostico multi-paso (autoregresivo/recursivo) para 7, 14, 30 y 60 dias
HORIZONTES = [7, 14, 30, 60]

def pronostico_recursivo(modelo, ultima_ventana_norm, pasos, scaler):
    """Toma la ULTIMA ventana de 60 dias reales (normalizada) y genera un
    pronostico recursivo: en cada paso predice el valor siguiente, lo agrega
    al final de la ventana y descarta el valor mas antiguo, de modo que cada
    nueva prediccion se apoya en las predicciones anteriores del propio modelo.
    Devuelve las predicciones desnormalizadas en USD.
    """
    ventana = ultima_ventana_norm.copy().reshape(1, VENTANA, 1)
    predicciones_norm = []

    for _ in range(pasos):
        pred = modelo.predict(ventana, verbose=0)[0, 0]
        predicciones_norm.append(pred)
        ventana = np.append(ventana[:, 1:, :], [[[pred]]], axis=1)

    predicciones_norm = np.array(predicciones_norm).reshape(-1, 1)
    predicciones_usd = scaler.inverse_transform(predicciones_norm).flatten()
    return predicciones_usd

print(f"Funcion pronostico_recursivo definida para los horizontes {HORIZONTES}")

Funcion pronostico_recursivo definida para los horizontes [7, 14, 30, 60]


In [10]:
# Paso 10 -- Bandas de confianza (Prediccion +/- 1.96*RMSE) y modelo ARIMA(1,1,1) de control
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

def calcular_bandas(predicciones_usd, rmse_usd, z=1.96):
    """Calcula la banda superior e inferior de cada prediccion:
    Prediccion +/- (z * RMSE), con z=1.96 para un intervalo de confianza ~95%.
    """
    banda_superior = predicciones_usd + z * rmse_usd
    banda_inferior = predicciones_usd - z * rmse_usd
    return banda_superior, banda_inferior


def rmse_arima_control(train_close, test_close):
    """Ajusta un ARIMA(1,1,1) simple sobre el train y pronostica len(test)
    pasos, como contraste de control y validacion de calidad frente al RMSE
    obtenido por la red LSTM.
    """
    try:
        modelo_arima = ARIMA(train_close, order=(1, 1, 1))
        ajuste = modelo_arima.fit()
        pronostico = ajuste.forecast(steps=len(test_close))
        rmse = float(np.sqrt(mean_squared_error(test_close, pronostico)))
        return rmse
    except Exception as e:
        print(f"    [ARIMA] No se pudo ajustar el modelo de control: {e}")
        return None

print("Funciones de bandas de confianza y ARIMA de control definidas")

Funciones de bandas de confianza y ARIMA de control definidas


In [11]:
# Paso 11 -- Metricas de evaluacion del modelo LSTM en el set de prueba
from sklearn.metrics import mean_absolute_error, r2_score

def calcular_metricas(y_test_usd, y_pred_usd):
    """Calcula RMSE (USD), RMSE (%) relativo al precio medio de test, MAE y R^2."""
    rmse_usd = float(np.sqrt(mean_squared_error(y_test_usd, y_pred_usd)))
    precio_medio = float(np.mean(y_test_usd))
    rmse_pct = float(rmse_usd / precio_medio * 100) if precio_medio else None
    mae = float(mean_absolute_error(y_test_usd, y_pred_usd))
    r2 = float(r2_score(y_test_usd, y_pred_usd))
    return {
        "rmse_usd": round(rmse_usd, 4),
        "rmse_pct": round(rmse_pct, 2) if rmse_pct is not None else None,
        "mae_usd": round(mae, 4),
        "r2": round(r2, 4)
    }

print("Funcion calcular_metricas definida")

Funcion calcular_metricas definida


In [12]:
# Paso 12 -- Pipeline completo por ticker: preprocesar, entrenar, pronosticar, evaluar y guardar
def procesar_ticker_lstm(ticker):
    """Ejecuta el pipeline completo del LSTM Regressor para un ticker:

    1) Garantiza datos en MongoDB (consulta existente o descarga con yfinance)
    2) Preprocesa (particion 80/20, normalizacion, ventanas de 60 dias)
    3) Construye y entrena el modelo LSTM
    4) Genera pronostico recursivo para 7/14/30/60 dias con bandas de confianza
    5) Calcula metricas y las contrasta con un ARIMA(1,1,1) de control
    6) Persiste predicciones y metricas en MongoDB

    Devuelve un diccionario con el resultado consolidado del ticker.
    """
    garantizar_datos_ticker(ticker)

    df = cargar_serie_close(ticker)
    if len(df) < VENTANA + 20:
        raise ValueError(f"Datos insuficientes para generar ventanas de {VENTANA} dias")

    serie_close = df["close"].values.astype(float)
    fecha_ultima = df["fecha"].iloc[-1]

    train_close, test_close = particionar_train_test(serie_close, prop_train=0.8)
    train_norm, test_norm, scaler = normalizar(train_close, test_close)

    # Se antepone la cola del train (ultimos VENTANA valores normalizados) al
    # test para poder generar ventanas completas tambien en las primeras filas
    # del propio set de prueba, sin perder muestras de evaluacion.
    serie_test_con_contexto = np.concatenate([train_norm[-VENTANA:], test_norm])

    X_train, y_train = generar_ventanas(train_norm, VENTANA)
    X_test, y_test = generar_ventanas(serie_test_con_contexto, VENTANA)

    modelo = construir_modelo_lstm(VENTANA)
    entrenar_modelo_lstm(modelo, X_train, y_train, X_test, y_test, epocas=100, batch_size=32)

    # Predicciones sobre el set de prueba, desnormalizadas a USD
    y_pred_test_norm = modelo.predict(X_test, verbose=0)
    y_pred_test_usd = scaler.inverse_transform(y_pred_test_norm).flatten()
    y_test_usd = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

    metricas = calcular_metricas(y_test_usd, y_pred_test_usd)
    rmse_arima = rmse_arima_control(train_close, test_close)

    # Pronostico multi-paso a partir de la ULTIMA ventana real de 60 dias
    ultima_ventana_norm = scaler.transform(serie_close[-VENTANA:].reshape(-1, 1))
    horizonte_max = max(HORIZONTES)
    predicciones_max = pronostico_recursivo(modelo, ultima_ventana_norm, horizonte_max, scaler)
    banda_sup_max, banda_inf_max = calcular_bandas(predicciones_max, metricas["rmse_usd"])

    predicciones_por_horizonte = {}
    for h in HORIZONTES:
        fechas_h = [fecha_ultima + timedelta(days=i) for i in range(1, h + 1)]
        predicciones_por_horizonte[h] = [
            {
                "fecha_proyectada": fechas_h[i].strftime("%Y-%m-%d"),
                "precio_proyectado_usd": limpiar_valor(predicciones_max[i]),
                "banda_superior": limpiar_valor(banda_sup_max[i]),
                "banda_inferior": limpiar_valor(banda_inf_max[i])
            }
            for i in range(h)
        ]

    # --- Persistencia en MongoDB Atlas ---
    # Coleccion predicciones_lstm: se borra el registro previo del ticker
    # para evitar duplicados al re-ejecutar el notebook.
    db["predicciones_lstm"].delete_many({"ticker": ticker})
    db["predicciones_lstm"].insert_one({
        "ticker": ticker,
        "modelo": "LSTM_Regressor",
        "fecha_base": fecha_ultima.strftime("%Y-%m-%d"),
        "horizontes": {str(h): predicciones_por_horizonte[h] for h in HORIZONTES},
        "created_at": datetime.now()
    })

    # Coleccion metricas_lstm: metricas del modelo + comparacion frente a ARIMA
    db["metricas_lstm"].delete_many({"ticker": ticker})
    db["metricas_lstm"].insert_one({
        "ticker": ticker,
        "modelo": "LSTM_Regressor",
        **metricas,
        "rmse_arima_control": round(rmse_arima, 4) if rmse_arima is not None else None,
        "mejora_vs_arima_pct": (
            round((rmse_arima - metricas["rmse_usd"]) / rmse_arima * 100, 2)
            if rmse_arima else None
        ),
        "muestras_train": int(len(X_train)),
        "muestras_test": int(len(X_test)),
        "fecha_entrenamiento": datetime.now()
    })

    rmse_usd = metricas["rmse_usd"]
    rmse_pct = metricas["rmse_pct"]
    mae_usd = metricas["mae_usd"]
    r2 = metricas["r2"]

    if rmse_arima is not None:
        print(f"  [{ticker}] RMSE=${rmse_usd:.2f} ({rmse_pct:.1f}%) | "
              f"MAE=${mae_usd:.2f} | R2={r2:.3f} | "
              f"RMSE ARIMA=${rmse_arima:.2f} | guardado en MongoDB")
    else:
        print(f"  [{ticker}] RMSE=${rmse_usd:.2f} ({rmse_pct:.1f}%) | "
              f"MAE=${mae_usd:.2f} | R2={r2:.3f} | guardado en MongoDB")

    return {
        "ticker": ticker,
        "metricas": metricas,
        "rmse_arima_control": rmse_arima,
        "predicciones": {str(h): predicciones_por_horizonte[h] for h in HORIZONTES}
    }

print("Funcion procesar_ticker_lstm definida")

Funcion procesar_ticker_lstm definida


In [13]:
# Paso 13 -- Procesar los 5 tickers del proyecto
print("Entrenando LSTM Regressor para los 5 tickers...")
print()

resultados_lstm = {}
for ticker in TICKERS.keys():
    try:
        resultados_lstm[ticker] = procesar_ticker_lstm(ticker)
    except Exception as e:
        print(f"  [{ticker}] Error: {e}")

print()
print(f"Procesamiento completo: {len(resultados_lstm)}/{len(TICKERS)} tickers procesados")

Entrenando LSTM Regressor para los 5 tickers...

  [FSM] 251 registros ya disponibles en MongoDB
  [FSM] RMSE=$0.59 (6.4%) | MAE=$0.48 | R2=0.363 | RMSE ARIMA=$0.87 | guardado en MongoDB
  [VOLCABC1.LM] 247 registros ya disponibles en MongoDB
  [VOLCABC1.LM] RMSE=$0.08 (9.4%) | MAE=$0.06 | R2=-0.656 | RMSE ARIMA=$0.11 | guardado en MongoDB
  [ABX.TO] 252 registros ya disponibles en MongoDB
  [ABX.TO] RMSE=$3.38 (6.1%) | MAE=$2.64 | R2=-0.102 | RMSE ARIMA=$4.50 | guardado en MongoDB
  [BVN] 251 registros ya disponibles en MongoDB
  [BVN] RMSE=$2.12 (6.5%) | MAE=$1.74 | R2=0.353 | RMSE ARIMA=$2.64 | guardado en MongoDB
  [BHP] 251 registros ya disponibles en MongoDB
  [BHP] RMSE=$9.46 (11.1%) | MAE=$8.38 | R2=-4.750 | RMSE ARIMA=$6.90 | guardado en MongoDB

Procesamiento completo: 5/5 tickers procesados


In [14]:
# Paso 14 -- Tabla de metricas finales del LSTM Regressor (set de prueba)
filas = []
for ticker, resultado in resultados_lstm.items():
    m = resultado["metricas"]
    filas.append({
        "Ticker": ticker,
        "RMSE (USD)": m["rmse_usd"],
        "RMSE (%)": m["rmse_pct"],
        "MAE (USD)": m["mae_usd"],
        "R2": m["r2"],
        "RMSE ARIMA (USD)": resultado["rmse_arima_control"]
    })

tabla_metricas = pd.DataFrame(filas).set_index("Ticker")
print("Tabla de metricas del LSTM Regressor (set de prueba):")
display(tabla_metricas.round(4))

Tabla de metricas del LSTM Regressor (set de prueba):


,RMSE (USD),RMSE (%),MAE (USD),R2,RMSE ARIMA (USD)
Ticker,,,,,
FSM,0.5927,6.43,0.4836,0.3631,0.8718
VOLCABC1.LM,0.0757,9.37,0.0633,-0.6561,0.1057
ABX.TO,3.3767,6.06,2.6386,-0.1023,4.5002
BVN,2.1205,6.48,1.7352,0.3535,2.6374
BHP,9.4557,11.13,8.3829,-4.7503,6.8955


In [15]:
# Paso 15 -- Consolidar los 5 tickers y exportar a datos_lstm_reg.json
def limpiar_json(obj):
    """Recorre recursivamente la estructura reemplazando NaN/Infinity por None,
    para garantizar que el JSON resultante sea siempre valido.
    """
    if isinstance(obj, dict):
        return {k: limpiar_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [limpiar_json(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)):
        return None
    return obj

consolidado = {
    "proyecto": "Ernesto Investing AI",
    "modelo": "LSTM_Regressor",
    "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "horizontes_dias": HORIZONTES,
    "tickers": {
        ticker: {
            "metricas": resultado["metricas"],
            "rmse_arima_control": resultado["rmse_arima_control"],
            "predicciones": resultado["predicciones"]
        }
        for ticker, resultado in resultados_lstm.items()
    }
}

consolidado = limpiar_json(consolidado)

RUTA_JSON = "datos_lstm_reg.json"
with open(RUTA_JSON, "w", encoding="utf-8") as f:
    json.dump(consolidado, f, ensure_ascii=False, indent=2)

nombre_archivo = RUTA_JSON
n_tickers = len(consolidado["tickers"])
print(f"Archivo '{nombre_archivo}' generado con {n_tickers} tickers")

Archivo 'datos_lstm_reg.json' generado con 5 tickers


In [16]:
# Paso 16 -- Celda de verificacion final
import os

print("Verificando persistencia en MongoDB Atlas y exportacion JSON...")
print()

resumen_ok = True

for ticker in TICKERS.keys():
    tiene_pred = db["predicciones_lstm"].count_documents({"ticker": ticker}) > 0
    tiene_met = db["metricas_lstm"].count_documents({"ticker": ticker}) > 0
    if tiene_pred and tiene_met:
        print(f"  [OK] {ticker}: predicciones y metricas guardadas en MongoDB")
    else:
        print(f"  [ATENCION] {ticker}: faltan datos en MongoDB (predicciones={tiene_pred}, metricas={tiene_met})")
        resumen_ok = False

print()
if os.path.exists(RUTA_JSON):
    tam = os.path.getsize(RUTA_JSON)
    print(f"  [OK] Archivo '{RUTA_JSON}' escrito correctamente ({tam} bytes)")
else:
    print(f"  [ATENCION] No se encontro el archivo '{RUTA_JSON}'")
    resumen_ok = False

print()
if resumen_ok:
    print("Verificacion exitosa: MongoDB Atlas y el archivo JSON estan sincronizados.")
else:
    print("Atencion: revisar los elementos marcados arriba antes de continuar.")

Verificando persistencia en MongoDB Atlas y exportacion JSON...

  [OK] FSM: predicciones y metricas guardadas en MongoDB
  [OK] VOLCABC1.LM: predicciones y metricas guardadas en MongoDB
  [OK] ABX.TO: predicciones y metricas guardadas en MongoDB
  [OK] BVN: predicciones y metricas guardadas en MongoDB
  [OK] BHP: predicciones y metricas guardadas en MongoDB

  [OK] Archivo 'datos_lstm_reg.json' escrito correctamente (108757 bytes)

Verificacion exitosa: MongoDB Atlas y el archivo JSON estan sincronizados.


## Resultado

Las colecciones `predicciones_lstm` y `metricas_lstm` contienen resultados reales del **LSTM Regressor** (pronosticos de precio en USD para los horizontes de 7, 14, 30 y 60 dias, con bandas de confianza al 95%) y sus metricas de evaluacion (RMSE, RMSE %, MAE, R2) contrastadas frente a un modelo **ARIMA(1,1,1)** de control, para los 5 tickers del proyecto.

El archivo `datos_lstm_reg.json` consolida estos resultados y queda listo para ser consumido directamente por el frontend.

**Pipeline:** yfinance -> MongoDB (`precios_ohlcv`) -> LSTM Regressor + ARIMA -> **MongoDB (`predicciones_lstm`, `metricas_lstm`) + `datos_lstm_reg.json`**